In [11]:
from phase_II.nifty_re_playground.strain_tools import *
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal.windows import tukey
from functools import partial
import os
import nifty.nifty.re as jft
from phase_II.nifty_re_playground.strain_tools import *
from phase_III.strain import *
import jax
from phase_III.strain.helpers import jft_model_vjp_jvp_stability
from phase_III.useful.helpers import plot_posterior
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
%matplotlib tk

key = jax.random.key(34)

## Plotting results from broken power law

In [ ]:
GW150914_bp = StrainSignalInference(
    key=key,
    event_name="GW150914",
    detector="H1",
    data_duration_of_hdf5_file="4096sec",
    stationarity_time_scale=32,
    e_fac=1,
    r_fac=1,
    alpha_taper_on_data=.1,
    out_name='bp'
)

broken_power_law = BrokenPowerLaw(
                            signal_grid=GW150914_bp.machinery.s_dom_real,
                            pl_slope_left=(1, .5),
                            peak_power=1e3,
                            sigmoid_width=30,
                            pl_slope_right=(-1, .5),
                            k_break=(10, 2000),
                            fluctuations=(1, 1),
                            envelope_fluctuations=(1, 1e-16),
                            envelope_loglogavgslope=(-4, 1),
                            )

GW150914_bp.add_signal_model(s_model=broken_power_law)
noise_cov_args = dict(one_sided_noise_ps=GW150914_bp.ps_welch, data_grid=GW150914_bp.machinery.d_dom_real)
N_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1), **noise_cov_args)
N_sqrt = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(1/2), **noise_cov_args)
N_sqrt_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1/2), **noise_cov_args)

posterior_latent_samples_bp, vi_info, key = GW150914_bp.run(kl_iterations=15, use_strict_minimizers=True)

In [ ]:
# Get data for plot
ps_op = broken_power_law.ps
ps_mean = jnp.mean(jnp.array([ps_op(sl) for sl in posterior_latent_samples_bp]), axis=0)
ps_std = jnp.mean(jnp.array([ps_op(sl) for sl in posterior_latent_samples_bp]), axis=0)

In [ ]:
# Construct
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))
ax_0: plt.Axes = axs[0]
ax_1: plt.Axes = axs[1]

# Fill upper plot
GW150914_bp.visualize_results(add_processed_data=True, custom_ax=axs[0], show=False, plot_oscillator_samples=False, plot_template=False)
# ax_0.set_xlim(-0.05, 0.075)
ax_0.set_xlim(-0.07, 0.08)  # fits to the axes of other plot
ax_0.set_ylabel(r"$h(t)$ $\mathrm{[10^{-19}]}$")

k_modes = GW150914_bp.machinery.k_signal
ax_1.loglog(k_modes, ps_mean, lw=2, color=blue, label=r"Posterior mean")
for sl in posterior_latent_samples_bp:
    ps = ps_op(sl)
    ax_1.loglog(k_modes, ps, lw=1, color=blue, alpha=0.1)

# ax_1.fill_between(x=k_modes, y1=ps_mean+ps_std, y2=ps_mean-ps_std, color=light_blue, alpha=0.7)

for ax in axs:
    ax.legend()

ax_0.text(0.025, 0.95,  # x, y in axes fraction (0–1)
            'H1',  # text
            transform=ax_0.transAxes,
            verticalalignment='top',
            horizontalalignment='left',
            fontsize=20,)

ax_0.set_xlabel(r"Time $t$ $\mathrm{[s]}$")
ax_1.set_xlabel(r"Frequency $f$ $\mathrm{[Hz]}$")
ax_1.set_ylabel(r"Power")
save_figure(tight_ly=True, show=True, save_fig=True)

## Plotting results from H1 oscillator model

In [12]:
GW150914_H1 = StrainSignalInference(
    key=key,
    event_name="GW150914",
    detector="H1",
    data_duration_of_hdf5_file="4096sec",
    stationarity_time_scale=32,
    e_fac=1,
    r_fac=1,
    alpha_taper_on_data=.1,
    out_name='osc'
)

signal_domain = GW150914_H1.machinery.t_ss
target_domain = GW150914_H1.machinery.t_ds
oscillator_prior_dct = {
    "frequency": {"offset_mean": 1000, "offset_std": (500, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (-2, 1e-16)},  # log fluctuations...
    "damping": {"offset_mean": 500, "offset_std": (250, 1e-16), "fluctuations": (1e-16, 1e-16), "loglogavgslope": (-2, 1e-16)},
    "force": {"offset_mean": 0, "offset_std": (1e-16, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (0, 1e-16), },
    "global_amplitude": (1, 1),
    "init_condition": (0, 0),
}
signal_prior = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain)
oscillator = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior)
# oscillator.plot_samples(20, key)
GW150914_H1.add_signal_model(s_model=oscillator)
noise_cov_args = dict(one_sided_noise_ps=GW150914_H1.ps_welch, data_grid=GW150914_H1.machinery.d_dom_real)
N_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1), **noise_cov_args)
N_sqrt = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(1/2), **noise_cov_args)
N_sqrt_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1/2), **noise_cov_args)
post_lat_samples_H1, vi_info, key = GW150914_H1.run(kl_iterations=12, use_strict_minimizers=True)


Assumed noise stationarity timescale for Welch-average:  32  seconds.
	Constructing 15 windows over which we average.
	Mean variance of tapered windows: 4.542402182156302
	Compare with area under welch ps:  4.545029553571364

Waveform query for GW150914. Found 1 local matche(s):
	0: 	IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5
Using  IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5


2026-02-22  22:15:26 PESummary WARNING : Could not find f_start in input file and one was not passed from the command line. Using 20.0Hz as default


Waveform model bank names:  C01:IMRPhenomXPHM C01:Mixed C01:SEOBNRv4PHM
Using  IMRPhenomXPHM  model.


/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/strain_tools/inference_scheme_re.py:572: UserWarning: self.add_noise_op() was not called by the user, using Gaussian noise with default variance level 1e-10.
  raise_warning(f"self.add_noise_op() was not called by the user, using Gaussian noise with default "
/Users/iason/PycharmProjects/STRAIN/nifty/nifty/re/model.py:164: UserWarning: drawing white parameters;
to silence this warning, overload the `init` method
  warn(msg)
overwriting `position_or_samples` with `resume`


Input gps center:  1126259462.4  maximum likelihood merger time from template:  1126259462.4241831

You are trying to set up the frequency prior for the oscillator with mean 1000.0±(500, 1e-16). To ensure positivity, 
we are exponentiating internally and therefore changing the mean and standard deviation values to 
 13.82±((Array(0.81, dtype=float64), Array(0., dtype=float64))).
If you are unsure this has the desired effect, check samples via `StochasticOscillatorPrior.plot_omega_samples`.
may I suggest the correction factor  0.9041455363552922

Initialing noise covariance based on a power spectrum with total area σ^2 = ∫ ps(k) dk ~ 4.865793180044668.
Applying callable lambda x: x**(-1)
may I suggest the correction factor  0.9041455363552922

Initialing noise covariance based on a power spectrum with total area σ^2 = ∫ ps(k) dk ~ 4.865793180044668.
Applying callable lambda x: x**(1/2)
may I suggest the correction factor  0.9041455363552922

Initialing noise covariance based on a power 

In [13]:
w = 10
h = 3 * 4  # height scales with number of plots

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(w, h), sharex=True, height_ratios=[h*0.4,h*0.2,h*0.2,h*0.2])

ax0: plt.Axes = axes[0]
ax1: plt.Axes = axes[1]
ax2: plt.Axes = axes[2]
ax3: plt.Axes = axes[3]

t = GW150914_H1.machinery.t_ss

# Fill first axis
GW150914_H1.visualize_results(add_processed_data=False, custom_ax=ax0,
                               show=False, plot_oscillator_samples=False,
                               plot_template=True)

# Fill rest step by step
omega = oscillator.omega
omega_mean, omega_std = jft.mean_and_std([omega(sl) for sl in post_lat_samples_H1])
ax1.plot(t, omega_mean, lw=2, color=blue)
ax1.fill_between(t, omega_mean - omega_std, omega_mean + omega_std, alpha=0.7, color=light_blue)

# Gamma
gamma = oscillator.gamma
gamma_mean, gamma_std = jft.mean_and_std([gamma(sl) for sl in post_lat_samples_H1])
ax2.plot(t, gamma_mean, lw=2, color=blue)
ax2.fill_between(t, gamma_mean - gamma_std, gamma_mean + gamma_std, alpha=0.7, color=light_blue)


# Force
force = oscillator.xi_force
force_mean, force_std = jft.mean_and_std([force(sl) for sl in post_lat_samples_H1])
ax3.plot(t, force_mean, lw=2, color=blue)
ax3.fill_between(t, force_mean - force_std, force_mean + force_std, alpha=0.7, color=light_blue)


# Other stuff
ax0.set_xlim(-0.07, 0.08)
ax0.set_ylim(-0.016, 0.016)
ax0.set_ylabel(r"$h(t)$ $\mathrm{[10^{-19}]}$")
ax0.legend(loc="lower left")

ax1.set_ylim(0, 1095)
ax2.set_ylim(200, 1400)

ax1.set_ylabel(r"$\omega(t)$")
ax2.set_ylabel(r"$\gamma(t)$")
ax3.set_ylabel(r"$\xi_f(t)$")

ax3.set_xlabel(r"Time $t$ $\mathrm{[s]}$")

ax0.text(0.025, 0.95,  # x, y in axes fraction (0–1)
            'H1',  # text
            transform=ax0.transAxes,
            verticalalignment='top',
            horizontalalignment='left',
            fontsize=20,)

save_figure(tight_ly=True, show=True, save_fig=False)

Posterior statistics:
Using alpha shape parameter of  0.0  for signal response


## Plotting results from L1 oscillator model

In [14]:
GW150914_L1 = StrainSignalInference(
    key=key,
    event_name="GW150914",
    detector="L1",
    data_duration_of_hdf5_file="4096sec",
    stationarity_time_scale=32,
    e_fac=1,
    r_fac=1,
    alpha_taper_on_data=.1,
    out_name='osc'
)

signal_domain = GW150914_L1.machinery.t_ss
# noinspection PyRedeclaration
target_domain = GW150914_L1.machinery.t_ds

oscillator_prior_dct = {
    "frequency": {"offset_mean": 1000, "offset_std": (500, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (-2, 1e-16)},  # log fluctuations...
    "damping": {"offset_mean": 500, "offset_std": (250, 1e-16), "fluctuations": (1e-16, 1e-16), "loglogavgslope": (-2, 1e-16)},
    "force": {"offset_mean": 0, "offset_std": (1e-16, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (0, 1e-16), },
    "global_amplitude": (1, 1),
    "init_condition": (0, 0),
}
signal_prior = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain)
oscillator = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior)
# oscillator.plot_samples(20, key)
GW150914_L1.add_signal_model(s_model=oscillator)
noise_cov_args = dict(one_sided_noise_ps=GW150914_L1.ps_welch, data_grid=GW150914_L1.machinery.d_dom_real)
N_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1), **noise_cov_args)
N_sqrt = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(1/2), **noise_cov_args)
N_sqrt_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1/2), **noise_cov_args)
post_lat_samples_L1, vi_info, key = GW150914_L1.run(kl_iterations=12, use_strict_minimizers=True)


Assumed noise stationarity timescale for Welch-average:  32  seconds.
	Constructing 15 windows over which we average.
	Mean variance of tapered windows: 5.142075748087426
	Compare with area under welch ps:  5.145035049514723

Waveform query for GW150914. Found 1 local matche(s):
	0: 	IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5
Using  IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5


2026-02-22  22:15:39 PESummary WARNING : Could not find f_start in input file and one was not passed from the command line. Using 20.0Hz as default


Waveform model bank names:  C01:IMRPhenomXPHM C01:Mixed C01:SEOBNRv4PHM
Using  IMRPhenomXPHM  model.


/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/strain_tools/inference_scheme_re.py:572: UserWarning: self.add_noise_op() was not called by the user, using Gaussian noise with default variance level 1e-10.
  raise_warning(f"self.add_noise_op() was not called by the user, using Gaussian noise with default "
/Users/iason/PycharmProjects/STRAIN/nifty/nifty/re/model.py:164: UserWarning: drawing white parameters;
to silence this warning, overload the `init` method
  warn(msg)
overwriting `position_or_samples` with `resume`


Input gps center:  1126259462.4  maximum likelihood merger time from template:  1126259462.417131

You are trying to set up the frequency prior for the oscillator with mean 1000.0±(500, 1e-16). To ensure positivity, 
we are exponentiating internally and therefore changing the mean and standard deviation values to 
 13.82±((Array(0.81, dtype=float64), Array(0., dtype=float64))).
If you are unsure this has the desired effect, check samples via `StochasticOscillatorPrior.plot_omega_samples`.
may I suggest the correction factor  1.0381167897590045

Initialing noise covariance based on a power spectrum with total area σ^2 = ∫ ps(k) dk ~ 5.586779332076787.
Applying callable lambda x: x**(-1)
may I suggest the correction factor  1.0381167897590045

Initialing noise covariance based on a power spectrum with total area σ^2 = ∫ ps(k) dk ~ 5.586779332076787.
Applying callable lambda x: x**(1/2)
may I suggest the correction factor  1.0381167897590045

Initialing noise covariance based on a power s

In [15]:
w = 10
h = 3 * 4  # height scales with number of plots

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(w, h), sharex=True, height_ratios=[h*0.4,h*0.2,h*0.2,h*0.2])

ax0: plt.Axes = axes[0]
ax1: plt.Axes = axes[1]
ax2: plt.Axes = axes[2]
ax3: plt.Axes = axes[3]

t = GW150914_L1.machinery.t_ss

# Fill first axis
GW150914_L1.visualize_results(add_processed_data=False, custom_ax=ax0,
                               show=False, plot_oscillator_samples=False,
                               plot_template=True)

# Fill rest step by step
omega = oscillator.omega
omega_mean, omega_std = jft.mean_and_std([omega(sl) for sl in post_lat_samples_L1])
ax1.plot(t, omega_mean, lw=2, color=blue)
ax1.fill_between(t, omega_mean - omega_std, omega_mean + omega_std, alpha=0.7, color=light_blue)

# Gamma
gamma = oscillator.gamma
gamma_mean, gamma_std = jft.mean_and_std([gamma(sl) for sl in post_lat_samples_L1])
ax2.plot(t, gamma_mean, lw=2, color=blue)
ax2.fill_between(t, gamma_mean - gamma_std, gamma_mean + gamma_std, alpha=0.7, color=light_blue)


# Force
force = oscillator.xi_force
force_mean, force_std = jft.mean_and_std([force(sl) for sl in post_lat_samples_L1])
ax3.plot(t, force_mean, lw=2, color=blue)
ax3.fill_between(t, force_mean - force_std, force_mean + force_std, alpha=0.7, color=light_blue)


# Other stuff
ax0.set_xlim(-0.07, 0.08)
ax0.set_ylim(-0.016, 0.016)
ax0.set_ylabel(r"$h(t)$ $\mathrm{[10^{-19}]}$")
ax0.legend(loc="lower left")

ax1.set_ylim(0, 1095)
ax2.set_ylim(200, 1400)

ax1.set_ylabel(r"$\omega(t)$")
ax2.set_ylabel(r"$\gamma(t)$")
ax3.set_ylabel(r"$\xi_f(t)$")

ax3.set_xlabel(r"Time $t$ $\mathrm{[s]}$")

ax0.text(0.025, 0.95,  # x, y in axes fraction (0–1)
            'L1',  # text
            transform=ax0.transAxes,
            verticalalignment='top',
            horizontalalignment='left',
            fontsize=20,)

save_figure(tight_ly=True, show=True, save_fig=False)

Posterior statistics:
Using alpha shape parameter of  0.0  for signal response


# Wigner function of H1 vs L1

In [16]:
xi_H1 = GW150914_H1.event_data.event_strain_white

xi_L1_raw = GW150914_L1.event_data.event_strain
xi_L1_raw = xi_L1_raw-np.mean(xi_L1_raw)
welch_amp = np.sqrt(GW150914_L1.ps_welch[GW150914_L1.machinery.d_h_dom_expander])
xi_L1 = whiten(y=xi_L1_raw, amp=welch_amp)

In [17]:
plt.plot(xi_L1)

In [18]:
Stress_H1, t_dual_H1, f_dual_H1 = Stress_jft(xi_H1, time=GW150914_H1.machinery.t_ss)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (9.779941412875802e-17) 


In [19]:
Stress_L1, t_dual_L1, f_dual_L1 = Stress_jft(xi_L1, time=GW150914_L1.machinery.t_ss)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (8.195960922184829e-17) 


In [20]:

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), sharex=True, sharey=True)

ax0: plt.Axes = axs[0]
ax1: plt.Axes = axs[1]

cmap = 'seismic'

cb0, im0 = visualize_stress(Stress_H1, rows=f_dual_H1, cols=t_dual_H1, smooth=True, custom_ax=ax0, delay_plot=True, colorbar_label='', return_aux=True, plot_colorbar=True, cmap=cmap)
cb1, im1 = visualize_stress(Stress_L1, rows=f_dual_L1, cols=t_dual_L1, smooth=True, custom_ax=ax1, delay_plot=True,  return_aux=True, cmap=cmap)

ax0.set_xlabel(r"Time $t$ $\mathrm{[s]}$")
ax1.set_xlabel(r"Time $t$ $\mathrm{[s]}$")
ax0.set_ylabel(r"Frequency $f$ $\mathrm{[Hz]}$")

ax0.set_xlim(-0.07, 0.08)
ax0.set_ylim(-400, 400)

# Further connect colorbars together
min_data = -110
max_data = 170

im0.set_clim(min_data, max_data)
cb0.update_normal(im0)

im1.set_clim(min_data, max_data)
cb1.update_normal(im1)

cb0.remove()
fig.subplots_adjust(right=0.85, bottom=0.2)

labels = ["H1", "L1"]

for ax, lab in zip((ax0, ax1), labels):
    ax.text(
        0.05, 0.95, lab,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=label_fontsize_pts,
        bbox=dict(facecolor='white', edgecolor='white', alpha=0.6, lw=0)

    )


save_figure(tight_ly=False, show=True, save_fig=True)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


## Coupling of $\xi_f$ to frequency

In [ ]:
GW150914_H1 = StrainSignalInference(
    key=key,
    event_name="GW150914",
    detector="H1",
    data_duration_of_hdf5_file="4096sec",
    stationarity_time_scale=32,
    e_fac=1,
    r_fac=1,
    alpha_taper_on_data=.1,
    out_name='osc'
)

signal_domain = GW150914_H1.machinery.t_ss
target_domain = GW150914_H1.machinery.t_ds
oscillator_prior_dct = {
    "frequency": {"offset_mean": 1000, "offset_std": (500, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (-2, 1e-16)},  # log fluctuations...
    "damping": {"offset_mean": 500, "offset_std": (250, 1e-16), "fluctuations": (1e-16, 1e-16), "loglogavgslope": (-2, 1e-16)},
    "force": {"offset_mean": 0, "offset_std": (1e-16, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (0, 1e-16), },
    "global_amplitude": (1, 1),
    "init_condition": (0, 0),
}
signal_prior = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain)
oscillator = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior)
# oscillator.plot_samples(20, key)
GW150914_H1.add_signal_model(s_model=oscillator)
noise_cov_args = dict(one_sided_noise_ps=GW150914_H1.ps_welch, data_grid=GW150914_H1.machinery.d_dom_real)
N_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1), **noise_cov_args)
N_sqrt = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(1/2), **noise_cov_args)
N_sqrt_inv = NoiseCovarianceFromPs(callable_to_apply=lambda x: x**(-1/2), **noise_cov_args)
post_lat_samples_H1, vi_info, key = GW150914_H1.run(kl_iterations=13, use_strict_minimizers=True)

In [ ]:

t = GW150914_H1.machinery.t_ss
rnd_seed = 0
w = 10
h = 3*3

# force with coupling and localization
signal_prior = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain, localize_force=(-0.244, 1e-16))
oscillator = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior)
force_1 = oscillator.xi_force
omega = oscillator.omega

# force without coupling and without localization
oscillator_prior_dct_large_force = {
    "frequency": {"offset_mean": 1000, "offset_std": (500, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (-2, 1e-16)},  # log fluctuations...
    "damping": {"offset_mean": 500, "offset_std": (250, 1e-16), "fluctuations": (1e-16, 1e-16), "loglogavgslope": (-2, 1e-16)},
    "force": {"offset_mean": 0, "offset_std": (1e-16, 1e-16), "fluctuations": (1e7/2, 1e6), "loglogavgslope": (0, 1e-16), },
    "global_amplitude": (1, 1),
    "init_condition": (0, 0),
}
signal_prior_2 = StochasticOscillatorPrior(oscillator_prior_dct_large_force, signal_time_domain=signal_domain, localize_force=None, couple_force_to_frequency=False)
oscillator_2 = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior_2)
force_2 = oscillator_2.xi_force

# force with only coupling
signal_prior_3 = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain, localize_force=None, couple_force_to_frequency=True)
oscillator_3 = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior_3)
force_3 = oscillator_3.xi_force

key, subkey = jax.random.split(jax.random.PRNGKey(rnd_seed), 2)
latent_sample = jft.random_like(key=subkey, primals=list(post_lat_samples_H1)[0]._tree)
omega_sl = omega(latent_sample)
force1_sl = force_1(latent_sample)
force2_sl = force_2(latent_sample)
force3_sl = force_3(latent_sample)
waveform1_sl = oscillator(latent_sample)
waveform2_sl = oscillator_2(latent_sample)
waveform3_sl = oscillator_3(latent_sample)


import matplotlib.pyplot as plt

# Colors
green = "#2ca02c"      # for omega
light_red = "#ff7f0e"  # for waveform
blue = "#1f77b4"       # for force variations
orange = "#ff7f0e"


fig, axes = plt.subplots(
    nrows=3, ncols=3, figsize=(w, h),
    sharex=True, sharey=False  # will override sharey per row
)

# First row: omega
for col in range(3):
    axes[0, col].plot(t, omega(latent_sample), lw=1, color=green)
    if col != 0:
        axes[0, col].set_yticklabels([])  # hide y labels
axes[0,0].set_ylabel(r"$\omega(t)$")

# Second row: forces
forces = [force2_sl*1e-7, force3_sl*1e-7, force1_sl*1e-7]
for col in range(3):
    axes[1, col].plot(t, forces[col], lw=1, color=blue)
    if col != 0:
        axes[1, col].set_yticklabels([])  # hide y labels
axes[1,0].set_ylabel(r"$\xi_f(t)\cdot 10^{-7}$")

# Third row: waveforms
waveforms = [waveform2_sl, waveform3_sl, waveform1_sl]
for col in range(3):
    axes[2, col].plot(t, waveforms[col], lw=1, color=light_red)
    if col != 0:
        axes[2, col].set_yticklabels([])  # hide y labels
axes[2,0].set_ylabel(r"$h(t)$")  # only first column
# keep y labels for all columns in last row
# axes[2,1].set_yticklabels([])  # leave visible if desired

# X labels only on last row
for col in range(3):
    axes[2, col].set_xlabel(r"Time $t$ [s]")

# Share y axes for each row
for col in range(0,3):
    axes[0,col].set_ylim(980, 5300)  # links y-axis to first column


for col in range(0,3):
    axes[1,col].set_ylim(-7, 7)  # links y-axis to first column

for col in range(0,3):
    axes[2,col].set_ylim(-14, 14)  # links y-axis to first column


x1 = .238
x2 = .513
x3 = .787
y0 = 0.3
axes[2,0].annotate(
    "",
    xytext=(x1, y0+0.1),
    xy=(x1, y0),
    xycoords='figure fraction',
    arrowprops=dict(arrowstyle="->", color="k", lw=3)
    )
axes[2,1].annotate(
    "",
    xytext=(x2, y0+0.1),
    xy=(x2, y0),
    xycoords='figure fraction',
    arrowprops=dict(arrowstyle="->", color="k", lw=3)
    )
axes[2,2].annotate(
    "",
    xytext=(x3, y0+0.1),
    xy=(x3, y0),
    xycoords='figure fraction',
    arrowprops=dict(arrowstyle="->", color="k", lw=3)
    )

axes[2,0].text(x1, 0.625, "+", ha='center', va='center', transform=fig.transFigure, fontsize=30)
axes[2,1].text(x2, 0.625, "+", ha='center', va='center', transform=fig.transFigure, fontsize=30)
axes[2,2].text(x3, 0.625, "+", ha='center', va='center', transform=fig.transFigure, fontsize=30)

axes[1,1].text(0.65, 0.9, "Frequency coupling", ha='center', va='center', transform=axes[1,1].transAxes, fontsize=10)
axes[1,2].text(0.5, 0.9, "Frequency coupling + localization", ha='center', va='center', transform=axes[1,2].transAxes, fontsize=10)

save_figure(tight_ly=False, show=True, save_fig=False)